# Multi-Head Attention

Multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions. Instead of performing a single attention function, the queries, keys, and values are projected into multiple subspaces, and attention is applied in parallel across these subspaces. The outputs are then concatenated and linearly transformed.


Multiple self-attention mechanisms are usually applied in parallel, and this is known as
multi-head self-attention. Now $H$ different sets of values, keys, and queries are computed:


\begin{eqnarray}
 \mathbf{V}_h &=& \boldsymbol\beta_{vh}\mathbf{1}^{T}+\boldsymbol\Omega_{\mathit{vh}}\mathbf{X}\nonumber \\
 \mathbf{Q}_h &=& \boldsymbol\beta_{qh}\mathbf{1}^{T}+\boldsymbol\Omega_{\mathit{qh}}\mathbf{X}\nonumber \\
 \mathbf{K}_h &=& \boldsymbol\beta_{kh}\mathbf{1}^{T}+\boldsymbol\Omega_{\mathit{kh}}\mathbf{X}.
 \end{eqnarray}

The $h$th self-attention mechanism or head can be written as:

\begin{eqnarray}
  \mbox{{\bf Sa}}_{h}[\mathbf{X}] =\mathbf{V}_h\cdot\mbox{\bf Softmax}\left[\frac{\mathbf{K}_h^{T}\mathbf{Q}_h}{\sqrt{D}_{q}}\right],
 \end{eqnarray}

where we have different parameters $\{\boldsymbol\beta_{vh},\boldsymbol\Omega_{vh}\}$, $\{\boldsymbol\beta_{qh},\boldsymbol\Omega_{qh}\}$, and $\{\boldsymbol\beta_{kh},\boldsymbol\Omega_{kh}\}$ for each head.
Typically, if the dimension of the inputs $\mathbf{x}_m$ is $D$ and there are $H$ heads, the values,
queries, and keys will all be of size $D/H$, as this allows for an eﬀicient implementation.
The outputs of these self-attention mechanisms are vertically concatenated, and another
linear transform $\boldsymbol\Omega_c$ is applied to combine them (figure 12.6):

<img src="assets/Chap12/TransformerBlockSAMultiHead.svg" style="filter: invert(1);" width="100%">

Multi-head self-attention. Self-attention occurs in parallel across
multiple “heads.” Each has its own queries, keys, and values. Here two heads are
depicted, in the cyan and orange boxes, respectively. The outputs are vertically
concatenated, and another linear transformation Ωc is used to recombine them.

\begin{eqnarray}
  \mbox{{\bf MhSa}}[\mathbf{X}] = \boldsymbol\Omega_{c}\Bigl[\mbox{{\bf Sa}}_{1}[\mathbf{X}]^T,\mbox{{\bf Sa}}_{2}[\mathbf{X}]^T,\ldots,\mbox{{\bf Sa}}_{H}[\mathbf{X}]^T \Bigr]^T.
 \end{eqnarray}

Multiple heads seem to be necessary to make self-attention work well. It has been
speculated that they make the self-attention network more robust to bad initializations.

Layer Normalization (LayerNorm) is a technique used in neural networks to keep the values of a vector on a consistent scale. It is particularly important in transformers because it helps stabilize training.

The intuition is simple: take a vector, subtract its mean, and divide by its standard deviation.



The multihead self-attention mechanism maps $N$ inputs $\mathbf{x_n} \in \mathbb{R}^D$ and returns N outputs $\mathbf{x^{\prime}} \in \mathbb{R}^D$
.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

# Set seed so we get the same random numbers
np.random.seed(3)
# Number of inputs
N = 6
# Number of dimensions of each input
D = 8
# Create an empty list
X = np.random.normal(size=(D,N))
# Print X
print(X)

[[ 1.789  0.437  0.096 -1.863 -0.277 -0.355]
 [-0.083 -0.627 -0.044 -0.477 -1.314  0.885]
 [ 0.881  1.71   0.05  -0.405 -0.545 -1.546]
 [ 0.982 -1.101 -1.185 -0.206  1.486  0.237]
 [-1.024 -0.713  0.625 -0.161 -0.769 -0.23 ]
 [ 0.745  1.976 -1.244 -0.626 -0.804 -2.419]
 [-0.924 -1.024  1.124 -0.132 -1.623  0.647]
 [-0.356 -1.743 -0.597 -0.589 -0.874  0.03 ]]


We'll use two heads. We'll need the weights and biases for the keys, queries, and values (equations 12.2 and 12.4). We'll use two heads, and (as in the figure), we'll make the queries keys and values of size D/H

In [ ]:
# Number of heads
H = 2
# QDV dimension
H_D = int(D/H)

# Set seed so we get the same random numbers
np.random.seed(0)

# Choose random values for the parameters for the first head
omega_q1 = np.random.normal(size=(H_D,D))
omega_k1 = np.random.normal(size=(H_D,D))
omega_v1 = np.random.normal(size=(H_D,D))
beta_q1 = np.random.normal(size=(H_D,1))
beta_k1 = np.random.normal(size=(H_D,1))
beta_v1 = np.random.normal(size=(H_D,1))

# Choose random values for the parameters for the second head
omega_q2 = np.random.normal(size=(H_D,D))
omega_k2 = np.random.normal(size=(H_D,D))
omega_v2 = np.random.normal(size=(H_D,D))
beta_q2 = np.random.normal(size=(H_D,1))
beta_k2 = np.random.normal(size=(H_D,1))
beta_v2 = np.random.normal(size=(H_D,1))

# Choose random values for the parameters
omega_c = np.random.normal(size=(D,D))

Now let's compute the multiscale self-attention

In [ ]:
# Define softmax operation that works independently on each column
def softmax_cols(data_in):
  # Exponentiate all of the values
  exp_values = np.exp(data_in) ;
  # Sum over columns
  denom = np.sum(exp_values, axis = 0);
  # Compute softmax (numpy broadcasts denominator to all rows automatically)
  softmax = exp_values / denom
  # return the answer
  return softmax


# Now let's compute multihead scaled self-attention in matrix form
def multihead_scaled_self_attention(X, omega_v1, omega_q1, omega_k1, beta_v1, beta_q1, beta_k1,
                                       omega_v2, omega_q2, omega_k2, beta_v2, beta_q2, beta_k2, omega_c):

  # Head 1: queries, keys, values (each H_D x N)
  Q1 = omega_q1 @ X + beta_q1
  K1 = omega_k1 @ X + beta_k1
  V1 = omega_v1 @ X + beta_v1
  # Scaled dot products (N x N): row m = key m, column n = query n
  # softmax_cols normalizes each column, i.e. over keys for each query
  head1 = V1 @ softmax_cols(K1.T @ Q1 / np.sqrt(Q1.shape[0]))

  # Head 2
  Q2 = omega_q2 @ X + beta_q2
  K2 = omega_k2 @ X + beta_k2
  V2 = omega_v2 @ X + beta_v2
  head2 = V2 @ softmax_cols(K2.T @ Q2 / np.sqrt(Q2.shape[0]))

  # Vertically concatenate heads (D x N) and recombine
  X_prime = omega_c @ np.vstack([head1, head2])

  return X_prime

# Run the self attention mechanism
X_prime = multihead_scaled_self_attention(X,omega_v1, omega_q1, omega_k1, beta_v1, beta_q1, beta_k1, omega_v2, omega_q2, omega_k2, beta_v2, beta_q2, beta_k2, omega_c)

# Print out the results
np.set_printoptions(precision=3)
print("Your answer:")
print(X_prime)

print("True values:")
print("[[-21.207  -5.373 -20.933  -9.179 -11.319 -17.812]")
print(" [ -1.995   7.906 -10.516   3.452   9.863  -7.24 ]")
print(" [  5.479   1.115   9.244   0.453   5.656   7.089]")
print(" [ -7.413  -7.416   0.363  -5.573  -6.736  -0.848]")
print(" [-11.261  -9.937  -4.848  -8.915 -13.378  -5.761]")
print(" [  3.548  10.036  -2.244   1.604  12.113  -2.557]")
print(" [  4.888  -5.814   2.407   3.228  -4.232   3.71 ]")
print(" [  1.248  18.894  -6.409   3.224  19.717  -5.629]]")

# If your answers don't match, then make sure that you are doing the scaling, and make sure the scaling value is correct

Your answer:
[[-21.207  -5.373 -20.933  -9.179 -11.319 -17.812]
 [ -1.995   7.906 -10.516   3.452   9.863  -7.24 ]
 [  5.479   1.115   9.244   0.453   5.656   7.089]
 [ -7.413  -7.416   0.363  -5.573  -6.736  -0.848]
 [-11.261  -9.937  -4.848  -8.915 -13.378  -5.761]
 [  3.548  10.036  -2.244   1.604  12.113  -2.557]
 [  4.888  -5.814   2.407   3.228  -4.232   3.71 ]
 [  1.248  18.894  -6.409   3.224  19.717  -5.629]]
True values:
[[-21.207  -5.373 -20.933  -9.179 -11.319 -17.812]
 [ -1.995   7.906 -10.516   3.452   9.863  -7.24 ]
 [  5.479   1.115   9.244   0.453   5.656   7.089]
 [ -7.413  -7.416   0.363  -5.573  -6.736  -0.848]
 [-11.261  -9.937  -4.848  -8.915 -13.378  -5.761]
 [  3.548  10.036  -2.244   1.604  12.113  -2.557]
 [  4.888  -5.814   2.407   3.228  -4.232   3.71 ]
 [  1.248  18.894  -6.409   3.224  19.717  -5.629]]
